# Exercises in Fairness in Machine Learning

## Exercise 1

For this exercise, we will use the `adult` dataset (available on moodle or from the [UCI Machine Learning repository](https://archive.ics.uci.edu/dataset/2/adult)). Do the following:

1. Load in the dataset and correct the error in the income column (replace the "." with the empty string such that there are only two categories).


In [ ]:
import pandas as pd

# load dataset
adult = pd.read_csv("adult.csv")

# remove the "." in income values
adult["income"] = adult["income"].str.replace(".", "", regex=False)

# check categories
adult["income"].unique()

array(['<=50K', '>50K'], dtype=object)

2. Create an X dataset using the variables "age", "workclass", "education", "occupation", "race", "sex", "hours-per-week". For the categorical variables with missing values, replace the missing values with a new category "Unknown". Also replace any values that are "?" with the value "Unknown" (using `str.replace`, for instance)


In [ ]:
X = adult[[
    "age",
    "workclass",
    "education",
    "occupation",
    "race",
    "sex",
    "hours-per-week"
]].copy()

# replace ? with Unknown
X = X.replace("?", "Unknown")

# replace NaN values
X = X.fillna("Unknown")

3. Turn the five categorical variables in X into dummy variables and remove the original five variables (This will probably give you around 44 columns in X)


In [ ]:
X = pd.get_dummies(X, drop_first=False)

4. Create the y variable by using the following code `y = pd.get_dummies(adult[["income"]], drop_first=True, dtype = "int", prefix="income"); y = y["income_>50K"].values`, where `adult` is the name of your dataframe you loaded the adult dataset into.


In [ ]:
y = pd.get_dummies(
    adult[["income"]],
    drop_first=True,
    dtype="int",
    prefix="income"
)

y = y["income_>50K"].values

5. Do a train-test split with 30% of the data for test (using `random_state=123`) and train a `GradientBoostingClassifier` model on the data.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=123
)

model = GradientBoostingClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

6. Evaluate your model using various evaluation metrics and look at the confusion matrix of your model


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.822425441889033
Precision: 0.7049559981472904
Recall: 0.4364783481502724
F1: 0.5391427559334042
[[10529   637]
 [ 1965  1522]]


7. To be able to calculate the various fairness metrics in regard to the variable `sex`, we need to construct two separate confusion matrices for the test dataset, one for `female` and one for `male`. First, create separate test sets for `female` and `male` as well as the predicted values for each gender. That is, create `X_test_female`, `X_test_male`, `y_test_female`, `y_test_male`, `y_pred_female`, and `y_pred_male`. You can create `X_test_female` by `X_test_female = X_test[X_test["sex_Male"] == 0]` and `y_test_male` by `y_test_male = y_test[X_test["sex_Male"] == 1]`, for instance.


In [ ]:
X_test_female = X_test[X_test["sex_Male"] == 0]
X_test_male   = X_test[X_test["sex_Male"] == 1]

y_test_female = y_test[X_test["sex_Male"] == 0]
y_test_male   = y_test[X_test["sex_Male"] == 1]

y_pred_female = model.predict(X_test_female)
y_pred_male   = model.predict(X_test_male)

8. We can now create the True Positive (TP), True Negative (TN), False Positive (FP), and False Negative (FN) for each gender. For instance, we can calculate the False Positive for female (FP_f) by `FP_f = sum((y_test_female == 0) & (y_pred_female == 1))`. Calculate the eight values `TP_f`, `TN_f`, `FP_f`, `FN_f`, `TP_m`, `TN_m`, `FP_m`, and `FN_m`.


In [ ]:
TP_f = sum((y_test_female == 1) & (y_pred_female == 1))
TN_f = sum((y_test_female == 0) & (y_pred_female == 0))
FP_f = sum((y_test_female == 0) & (y_pred_female == 1))
FN_f = sum((y_test_female == 1) & (y_pred_female == 0))

TP_m = sum((y_test_male == 1) & (y_pred_male == 1))
TN_m = sum((y_test_male == 0) & (y_pred_male == 0))
FP_m = sum((y_test_male == 0) & (y_pred_male == 1))
FN_m = sum((y_test_male == 1) & (y_pred_male == 0))

9. Calculate the accuracy for female and male and comment on the results


In [ ]:
acc_f = (TP_f + TN_f) / (TP_f + TN_f + FP_f + FN_f)
acc_m = (TP_m + TN_m) / (TP_m + TN_m + FP_m + FN_m)

print("Female accuracy:", acc_f)
print("Male accuracy:", acc_m)

Female accuracy: 0.9025545941491553
Male accuracy: 0.7827329319318298


10. Is there error rate balance, i.e. are the false positive rate (FPR) and false negative rate (FNR) the same across the two genders?


In [ ]:
FPR_f = FP_f / (FP_f + TN_f)
FPR_m = FP_m / (FP_m + TN_m)

FNR_f = FN_f / (FN_f + TP_f)
FNR_m = FN_m / (FN_m + TP_m)

11. Is there predictive parity?


In [ ]:
precision_f = TP_f / (TP_f + FP_f)
precision_m = TP_m / (TP_m + FP_m)

12. Is there Statistical parity?


In [ ]:
stat_f = sum(y_pred_female == 1) / len(y_pred_female)
stat_m = sum(y_pred_male == 1) / len(y_pred_male)

15. [Discussion question] Can your Gradient Boosting Classifier be used to make fair salary predictions?


Reasons:

The model learns patterns from historical data

If historical salaries reflect gender inequality, the model reproduces that bias

Even if sex is removed, proxy variables (occupation, hours worked, education) can encode gender differences.

Therefore:

Gradient Boosting may produce accurate but unfair predictions.

Fairness methods would be required (reweighing, fairness constraints, etc.).

13. [Discussion question] In what sense is the `adult` dataset biased (unfair)?


The dataset is biased because:

Men historically earn higher salaries than women

Certain occupations are gender dominated

Women often have lower recorded hours or workforce participation

Therefore the dataset reflects historical labor market inequality.

14. [Discussion question] If the dataset is biased, where could the bias potentially come from?

Possible sources:

Historical societal bias

Gender wage gaps in the real world.

Sampling bias

The dataset represents 1994 US census data, which may not reflect balanced demographics.

Measurement bias

Variables like:

occupation

hours-per-week

education

can encode structural inequalities.

Label bias

Income itself reflects biased hiring and salary practices.